## LangGraph ReAct Agent with Tools
Learning Objectives:
- Create tools (functions) that LLMs can call
- Implement the ReAct pattern (Reasoning + Acting)
- Build an agent that decides when to use tools

#### 
Real-World Tools:
-----------------
- Database queries
- API calls (weather, stock prices, etc.)
- File operations
- Web searches
- Send emails/notifications
- Execute code
- Image generation
- Data analysis

### Agent Patterns
- Chain of Thoughts (CoT)
- Tree of Thoughts (ToT)
- ReAct

In [1]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START,END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
load_dotenv()

True

In [2]:
llm = ChatOpenAI(temperature=0) #  do experiment with different temperatures


In [3]:
import my_tools
# examples 
my_tools.calculate.invoke({'expression': '2+2*1.4/23-34'})

[TOOL] calculate ('2+2*1.4/23-34') -> '-31.878260869565217'


'-31.878260869565217'

In [4]:
eval('2+2*1.4/23-34')

-31.878260869565217

In [5]:
all_tools=[my_tools.calculate,my_tools.get_weather]

from typing_extensions import TypedDict, Annotated
import operator
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

In [20]:

def agent_node(state: AgentState):
    llm_with_tools = llm.bind_tools(all_tools)
    messages = state['messages']
    print('message output: ',messages)
    response = llm_with_tools.invoke(messages)
    print('response output: ',response)
    return {'messages': [response]}

In [22]:
state = {"messages": [HumanMessage("Hi")]}
result = agent_node(state)

message output:  [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={})]
response output:  content='Hello! How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 294, 'total_tokens': 304, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DUlrCfl64dfpVxWF1IcKJpgu34SVT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d8f54-8421-7a02-8066-9bfaf13bc1d0-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 294, 'output_tokens': 10, 'total_tokens': 304, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [17]:
result

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 299, 'total_tokens': 314, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DUlomPk690Sm0fBeCHVR0rjA5gaip', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8f52-3984-7671-8bed-04d33eb312f3-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'call_2Mwv7L0RqlKBNfe9qfsR7vSn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 299, 'output_tokens': 15, 'total_tokens': 314, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}

In [10]:
result["messages"][-1].content

''

In [8]:
result['messages']

[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 299, 'total_tokens': 314, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DUllS3o9mPMUtfAyGyuqF3g1zOqn4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8f4f-11ea-7643-b858-ac212b4aae7a-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'call_CoZrFR5PJfDUZa4f5oki3cDu', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 299, 'output_tokens': 15, 'total_tokens': 314, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

In [23]:
llm_with_tools = llm.bind_tools(all_tools)
llm_with_tools.invoke([HumanMessage("What is the weather in Mumbai?")])

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 300, 'total_tokens': 315, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DUlrygAbTUbhNIGFnshvplU7YxFrJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8f55-4110-74c3-b862-53bf45c8aebb-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'call_bwKq4mkMGtNWch5DlX3TNati', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 300, 'output_tokens': 15, 'total_tokens': 315, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [24]:
from typing import Annotated, Sequence
import operator
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from typing import TypedDict
import my_tools

In [25]:
# documentatio n code

# 1. State
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# 2. Tools & LLM
all_tools = [my_tools.calculate, my_tools.get_weather]
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
llm_with_tools = llm.bind_tools(all_tools)

# 3. Nodes
def agent_node(state: AgentState):
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(all_tools)

# 4. Routing function  ← KEY PART
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"

# 5. Graph wiring  ← mapping dict is REQUIRED
workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)
workflow.set_entry_point("agent")

workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", "end": END}  # ← mapping dict, not just lambda
)
workflow.add_edge("tools", "agent")

app = workflow.compile()

# 6. Run
result = app.invoke({
    "messages": [HumanMessage(content="What is the weather in Mumbai?")]
})

print(result["messages"][-1].content)

The current weather in Mumbai is haze with a temperature of 29°C (feels like 32°C) and humidity at 70%.
